---
## 0. How to Use This Notebook

1. Open this notebook **at the root of your Git repository**
2. Run cells top‑to‑bottom
3. Generated artifacts will appear in:
   ```
   docs/
     visuals/
     diagrams/
     lineage/
   ```
4. Drop outputs directly into a slide deck

---
## 1. Install Dependencies

> ⚠️ These commands are **system‑level installs**. Run once per machine.

### macOS
```bash
brew install gource ffmpeg
```

### Ubuntu / Debian
```bash
sudo apt update && sudo apt install -y gource ffmpeg
```

### Windows (Chocolatey)

If you don't have Chocolatey installed, install it first (run in PowerShell as Administrator):

```powershell
Set-ExecutionPolicy Bypass -Scope Process -Force
[System.Net.ServicePointManager]::SecurityProtocol = [System.Net.ServicePointManager]::SecurityProtocol -bor 3072
iex ((New-Object System.Net.WebClient).DownloadString('https://community.chocolatey.org/install.ps1'))
```

If Chocolatey is not in your PATH, add it:
```powershell
[Environment]::SetEnvironmentVariable("Path", $env:Path + ";C:\ProgramData\chocolatey\bin", [EnvironmentVariableTarget]::User)
```

Then install Gource and ffmpeg:
```powershell
choco install gource ffmpeg -y
```

---
## 2. Verify You Are in a Git Repository

In [1]:
import subprocess
import os

def run(cmd):
    """Run a shell command and return its output."""
    return subprocess.check_output(cmd, shell=True, text=True)

# Verify we're in a git repository
try:
    print(run("git status"))
    print("\n✅ Git repository verified!")
except subprocess.CalledProcessError:
    print("❌ Not in a Git repository. Please navigate to your repository root.")

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	viz-outputs/

nothing added to commit but untracked files present (use "git add" to track)


✅ Git repository verified!


---
## 3. Repository Archaeology (Extract the Signals)

### 3.1 Full Commit Timeline

In [2]:
# Get commit history
commit_log = run("git log --oneline --decorate --date=short --all -50")
print("Recent 50 commits:")
print(commit_log)

print("\n💡 Use this output to identify phases such as:")
print("   - Initial prototype")
print("   - Major refactor")
print("   - Automation")
print("   - Evaluation / benchmarking")

Recent 50 commits:
f77e55b (HEAD -> main, origin/main, origin/authorship-rewrite, authorship-rewrite) manuscript review and pruning old research files
1a40447 (origin/claude/review-manuscript-chapter-dn8Nz) Expand Chapter 2 literature review addressing key gaps
54c95e8 (origin/claude/review-manuscript-chapter-one-IlFQu) Address Chapter 1 gaps: add SE definitions, strengthen problem statement, complete failure modes section
2c9e5cf (origin/claude/create-research-prompts-ju8KB) Add dissertation-aware related work prompts for literature review
66c181d Add Chapter 2 literature review gap analysis
e161d87 Merge branch 'main' of https://github.com/ryan-a-bell/dissertation
2e26dec Merge branch 'main' of https://github.com/ryan-a-bell/dissertation
44f31ef Added viz/ directory for visualizing the repository and its development/evolution over time. Needs refinement.
5b1983a Added viz/ directory for visualizing the repository and its development/evolution over time. Needs refinement.
bdfdb4c Merg

### 3.2 Branch Inventory

In [3]:
# List all branches
branches = run("git branch --all")
print("All branches:")
print(branches)

print("\n💡 Take note of:")
print("   - Long‑lived branches")
print("   - Feature / experiment branches")
print("   - Release or paper branches")

All branches:
  authorship-rewrite
  claude/add-llm-judge-evaluation-01FAv1DGw7UNxPPAoRN5pyZi
  claude/cleanup-phase5-notebook-01UWUUAWiQaXuSDZkVcMUatc
  claude/consolidate-phase-6-files-01WoenvAEJVCTL5rMFqCXbfg
  claude/debug-auto-v2-files-01Eqtk4cnfCuV5RDUFdqWytr
  claude/debug-phase6-errors-01EdHYLcEDmXaP1xxaEd6wdV
  claude/debug-runpod-worker-01RvSsGKyXEeeT3HB6NdT6bd
  claude/document-phase6-analysis-016ujyVvbiZRnBMYs1gPHBCS
  claude/fix-beeswarm-plot-performance-01RKB5rM2gnjehCiXaSopZAx
  claude/fix-duplicate-model-size-01QychZ8D8Nt28r6Z85qjpF7
  claude/fix-json-formatting-01EN4QUsr1pmpokh5FYJANs2
  claude/fix-llm-judge-concurrency-014Vn8tiyNn4RnvYm54dFeV4
  claude/fix-model-pulling-notebook-01G8u8axsAwcQgij8uLU2YfB
  claude/llm-judge-runpod-auto-v2-01Wwv4SB3iUQ8dtqhxDqxuWL
  claude/organize-judge-models-prompts-01RfpuDkZpDMuwCnS8u6iNcg
  claude/phase6-analysis-results-011CV2eHEmAo5m1uUxUBRtua
  claude/plan-tokenomics-todos-01QzycG25w1apofDxB1a75da
  claude/review-llm-judge-manual

---
## 4. Quantitative Evidence (Credibility Layer)

In [4]:
# FIXED VERSION - Better subprocess handling for Windows
import time
import subprocess

def run_robust(cmd):
    """Run a shell command with proper Windows handling."""
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True,
        stdin=subprocess.DEVNULL,  # KEY FIX: Don't wait for stdin
        timeout=30
    )
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stderr)
    return result.stdout

print("🔍 Running repository metrics commands individually...\n")

# Command 1: Total commits
print("1️⃣ Counting total commits...")
start = time.time()
try:
    total_commits = int(run_robust("git rev-list --count HEAD").strip())
    print(f"   ✅ Done in {time.time() - start:.2f}s: {total_commits} commits\n")
except Exception as e:
    print(f"   ❌ Failed: {e}\n")
    total_commits = 0

# Command 2: Contributors (use HEAD to avoid stdin wait)
print("2️⃣ Getting contributors...")
start = time.time()
try:
    contributors_output = run_robust("git shortlog -sn HEAD")
    contributors = len(contributors_output.strip().split('\n'))
    print(f"   ✅ Done in {time.time() - start:.2f}s: {contributors} contributors\n")
    print(f"   Output:\n{contributors_output}")
except Exception as e:
    print(f"   ❌ Failed: {e}\n")
    contributors = 0

# Command 3: Branches
print("3️⃣ Counting branches...")
start = time.time()
try:
    branches_output = run_robust("git branch --all")
    branches = len(branches_output.strip().split('\n'))
    print(f"   ✅ Done in {time.time() - start:.2f}s: {branches} branches\n")
except Exception as e:
    print(f"   ❌ Failed: {e}\n")
    branches = 0

# Summary
print("="*60)
print("📊 Repository Metrics:")
print(f"   total_commits: {total_commits}")
print(f"   contributors: {contributors}")
print(f"   branches: {branches}")
print("="*60)

🔍 Running repository metrics commands individually...

1️⃣ Counting total commits...
   ✅ Done in 0.11s: 239 commits

2️⃣ Getting contributors...
   ✅ Done in 0.09s: 2 contributors

   Output:
   193	ryan-a-bell
    46	Claude

3️⃣ Counting branches...
   ✅ Done in 0.11s: 60 branches

📊 Repository Metrics:
   total_commits: 239
   contributors: 2
   branches: 60


---
## 5. Generate the Gource Animation (Core Artifact)

### 5.1 Prepare Output Directories

In [5]:
# Create output directories

# TODO: Create a sub-directory variable to make this parametric...

os.makedirs("viz-outputs", exist_ok=True)
print("✅ Output directory created: viz-outputs/")

✅ Output directory created: viz-outputs/


### 5.2 Run Gource + Export MP4

> 🎯 This produces a **slide‑ready video**

**Note:** This cell requires `gource` and `ffmpeg` to be installed and in your PATH.

In [5]:
import shutil
import os
import subprocess

# Check if required tools are in PATH
print("🔍 Checking prerequisites...")

# On Windows, refresh PATH from environment variables (like we did in PowerShell)
if os.name == 'nt':
    import winreg
    try:
        # Get Machine PATH
        with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r'SYSTEM\CurrentControlSet\Control\Session Manager\Environment') as key:
            machine_path = winreg.QueryValueEx(key, 'PATH')[0]
        
        # Get User PATH
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Environment') as key:
            user_path = winreg.QueryValueEx(key, 'PATH')[0]
        
        # Refresh PATH in current process
        os.environ['PATH'] = machine_path + ';' + user_path
        print("   🔄 Refreshed PATH from Windows environment variables")
    except Exception as e:
        print(f"   ⚠️ Could not refresh PATH: {e}")

# Try to find gource
gource_path = shutil.which("gource")

# Try to find ffmpeg
ffmpeg_path = shutil.which("ffmpeg")

if gource_path:
    print(f"   ✅ gource found: {gource_path}")
else:
    print("   ❌ gource NOT FOUND")
    print("      Install with: choco install gource -y")
    print("      After installing, re-run this cell (PATH will be refreshed)")

if ffmpeg_path:
    print(f"   ✅ ffmpeg found: {ffmpeg_path}")
else:
    print("   ❌ ffmpeg NOT FOUND")
    print("      Install with: choco install ffmpeg -y")
    print("      After installing, re-run this cell (PATH will be refreshed)")

if not gource_path or not ffmpeg_path:
    print("\n⚠️ Cannot proceed - missing required tools")
    print("\n💡 After installing them, just re-run this cell")
else:
    # Define output path
    output_dir = "viz-outputs"
    output_file = "repo_lineage.mp4"
    output_path = os.path.join(output_dir, output_file)
    output_abs_path = os.path.abspath(output_path)
    
    print(f"\n📁 Output will be saved to:")
    print(f"   {output_abs_path}")
    
    # Generate Gource animation using absolute paths
    gource_cmd = f'''"{gource_path}" --seconds-per-day 0.4 --auto-skip-seconds 1 --file-idle-time 0 --max-files 3000 --hide mouse,progress --title "Repository Evolution" --background-colour 000000 -1280x720 --output-ppm-stream - | "{ffmpeg_path}" -y -r 60 -f image2pipe -vcodec ppm -i - -vcodec libx264 -preset slow -crf 18 -pix_fmt yuv420p "{output_path}"'''

    print("\n🎬 Generating Gource animation...")
    print("⏳ This may take a few minutes depending on repository size...")
    
    try:
        result = subprocess.run(gource_cmd, shell=True, capture_output=True, text=True)
        
        # Check if file was actually created
        if os.path.exists(output_path):
            file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
            print("\n" + "="*60)
            print("✅ SUCCESS! Animation created successfully!")
            print("="*60)
            print(f"📹 File: {output_abs_path}")
            print(f"📊 Size: {file_size_mb:.2f} MB")
            print(f"💡 Open with: start {output_path}")
            print("="*60)
        else:
            print(f"\n❌ Error: File was not created at {output_path}")
            if result.returncode != 0:
                print(f"\nError output:")
                print(result.stderr)
            
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Make sure gource and ffmpeg are installed and in your PATH")

🔍 Checking prerequisites...
   🔄 Refreshed PATH from Windows environment variables
   ✅ gource found: C:\Program Files\Gource\cmd\gource.CMD
   ✅ ffmpeg found: C:\ProgramData\chocolatey\bin\ffmpeg.EXE

📁 Output will be saved to:
   c:\Users\rabel\Desktop\dissertation\viz\viz-outputs\repo_lineage.mp4

🎬 Generating Gource animation...
⏳ This may take a few minutes depending on repository size...

✅ SUCCESS! Animation created successfully!
📹 File: c:\Users\rabel\Desktop\dissertation\viz\viz-outputs\repo_lineage.mp4
📊 Size: 45.57 MB
💡 Open with: start viz-outputs\repo_lineage.mp4


---
## 6. Phase Commentary (Why the Repo Evolved)

In [ ]:
# Create output directory
os.makedirs("viz-outputs", exist_ok=True)
print("✅ Output directory created: viz-outputs/")

In [ ]:
# PARAMETRIC Phase Commentary - Generated from commit history analysis
import subprocess
from datetime import datetime
from collections import defaultdict
import re

def run_git(cmd):
    """Run a git command with proper Windows handling."""
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        stdin=subprocess.DEVNULL, timeout=60
    )
    return result.stdout.strip()

print("🔍 Analyzing commit history for phases...\n")

# Get all commits with date and message
commits_raw = run_git('git log --format="%ad|%s" --date=short HEAD')
commits = []
for line in commits_raw.split('\n'):
    if '|' in line:
        date, msg = line.split('|', 1)
        commits.append({'date': date, 'message': msg.lower()})

print(f"📊 Analyzing {len(commits)} commits...\n")

# Define keyword patterns for different phases
phase_patterns = {
    'setup': ['initial', 'setup', 'scaffold', 'init', 'first', 'create', 'add readme'],
    'exploration': ['experiment', 'try', 'test', 'explore', 'prototype', 'spike', 'wip', 'draft'],
    'core_development': ['implement', 'add', 'feature', 'build', 'create', 'develop'],
    'refactoring': ['refactor', 'clean', 'reorganize', 'restructure', 'improve', 'optimize'],
    'automation': ['automate', 'pipeline', 'ci', 'script', 'workflow', 'action', 'makefile'],
    'documentation': ['doc', 'readme', 'comment', 'explain', 'guide', 'tutorial'],
    'testing': ['test', 'benchmark', 'evaluate', 'measure', 'validate', 'assess'],
    'fixes': ['fix', 'bug', 'patch', 'correct', 'resolve', 'hotfix'],
    'release': ['release', 'version', 'publish', 'deploy', 'tag', 'merge']
}

# Categorize commits
phase_commits = defaultdict(list)
for commit in commits:
    msg = commit['message']
    for phase, keywords in phase_patterns.items():
        if any(kw in msg for kw in keywords):
            phase_commits[phase].append(commit)
            break

# Get date range
if commits:
    first_date = commits[-1]['date']
    last_date = commits[0]['date']
    print(f"📅 Repository timeline: {first_date} → {last_date}\n")

# Analyze by time periods (quarters)
commits_by_quarter = defaultdict(int)
for commit in commits:
    year_month = commit['date'][:7]  # YYYY-MM
    commits_by_quarter[year_month] += 1

# Generate phase analysis
print("="*60)
print("📈 COMMIT ACTIVITY BY CATEGORY:")
print("="*60)
for phase, phase_list in sorted(phase_commits.items(), key=lambda x: -len(x[1])):
    pct = len(phase_list) / len(commits) * 100
    bar = "█" * int(pct / 5)
    print(f"   {phase:20} {len(phase_list):3} commits ({pct:5.1f}%) {bar}")

# Generate the phases markdown
phases_md = f"""# Repository Evolution Phases

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}  
**Total Commits Analyzed:** {len(commits)}  
**Timeline:** {first_date if commits else 'N/A'} → {last_date if commits else 'N/A'}

---

## Commit Activity by Category

| Category | Commits | Percentage |
|----------|---------|------------|
"""

for phase, phase_list in sorted(phase_commits.items(), key=lambda x: -len(x[1])):
    pct = len(phase_list) / len(commits) * 100
    phases_md += f"| {phase.replace('_', ' ').title()} | {len(phase_list)} | {pct:.1f}% |\n"

phases_md += """
---

## Inferred Development Phases

"""

# Infer phases from temporal patterns
if commits:
    # Split commits into quarters of the timeline
    total = len(commits)
    q1 = commits[int(total*0.75):]  # Oldest 25%
    q2 = commits[int(total*0.5):int(total*0.75)]
    q3 = commits[int(total*0.25):int(total*0.5)]
    q4 = commits[:int(total*0.25)]  # Newest 25%
    
    def summarize_quarter(q_commits, name):
        if not q_commits:
            return ""
        date_range = f"{q_commits[-1]['date']} → {q_commits[0]['date']}"
        keywords = defaultdict(int)
        for c in q_commits:
            for phase, kws in phase_patterns.items():
                if any(kw in c['message'] for kw in kws):
                    keywords[phase] += 1
        top_activity = sorted(keywords.items(), key=lambda x: -x[1])[:3]
        activity_str = ", ".join([f"{k.replace('_', ' ')}" for k, v in top_activity])
        return f"""### {name}
**Period:** {date_range}  
**Commits:** {len(q_commits)}  
**Primary Activities:** {activity_str or 'General development'}

"""
    
    phases_md += summarize_quarter(q1, "Phase 1 – Foundation")
    phases_md += summarize_quarter(q2, "Phase 2 – Core Development")
    phases_md += summarize_quarter(q3, "Phase 3 – Expansion")
    phases_md += summarize_quarter(q4, "Phase 4 – Refinement")

phases_md += """---

## Monthly Commit Activity

"""
for month, count in sorted(commits_by_quarter.items()):
    bar = "█" * min(count, 30)
    phases_md += f"- **{month}**: {count} commits {bar}\n"

# Save to file with UTF-8 encoding
with open("viz-outputs/phases.md", "w", encoding="utf-8") as f:
    f.write(phases_md)

print("\n" + "="*60)
print("✅ Phase commentary written to viz-outputs/phases.md")
print("="*60)
print("\n📝 Summary:")
print(f"   - Analyzed {len(commits)} commits")
print(f"   - Identified {len([p for p in phase_commits if phase_commits[p]])} activity categories")
print(f"   - Generated 4 temporal phases")
print("\n💡 Edit viz-outputs/phases.md to add your own narrative!")

🔍 Analyzing commit history for phases...

📊 Analyzing 239 commits...

📅 Repository timeline: 2025-05-24 → 2025-12-29

📈 COMMIT ACTIVITY BY CATEGORY:
   core_development      68 commits ( 28.5%) █████
   release               30 commits ( 12.6%) ██
   documentation         27 commits ( 11.3%) ██
   fixes                 18 commits (  7.5%) █
   automation            15 commits (  6.3%) █
   setup                 13 commits (  5.4%) █
   exploration            9 commits (  3.8%) 
   refactoring            6 commits (  2.5%) 


UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 128: character maps to <undefined>